In [1]:
import pandas as pd
import numpy as np

In [2]:
# Loading the datasets
arise = pd.read_csv("ARIES_Dataset.csv")
civic = pd.read_csv("CivicPulse_KS.csv")

print("ARISE shape:", arise.shape)
print("CivicPulse shape:", civic.shape)

display(arise.head())
display(civic.head())

ARISE shape: (309, 91)
CivicPulse shape: (135, 74)


,Unnamed: 0,ResponseId,RecipientLastName,ExternalReference,1st Dis,2nd Dis,3rd Dis,Q3_10_Agri,Q3_9_Cyber,Q3_19_Dam,...,Q13_Reduce staff,Q13_Defer capital projects,Q13_Reduce benefits,Q13_Reduce services,Q13_Increase taxes,Q13_Increase user fees,Q13_Adopt new fees,Q13_Reduce fund balance,Q13_Defer maintenance expenditures,Q13_None of the above
0,1,R_2sQNn7iItA9jdBn,chief administrator,2079950,3.0,21.0,1.0,NaN,NaN,NaN,...,1,1,1,0,1,1,1,0,0,0
1,2,R_2Qyp1YFLCR17M9C,chief administrator,2061250,11.0,3.0,8.0,NaN,NaN,NaN,...,1,1,0,1,1,0,0,0,1,0
2,3,R_2aQAihYklmvMFey,chief administrator,2005600,11.0,21.0,1.0,NaN,NaN,NaN,...,1,1,0,1,1,0,0,0,1,0
3,4,R_6sTVkGKJ3zy6aPL,chief administrator,2034300,21.0,1.0,2.0,NaN,NaN,NaN,...,0,1,0,0,1,0,0,0,0,0
4,5,R_3sddVtnZ4CSdNk0,chief administrator,2053225,3.0,1.0,23.0,NaN,NaN,NaN,...,1,1,1,1,1,1,1,1,1,0


,Respondentid,statecode,countycode,countysubcode,placecode,govname,state,stpl_fips,stco_fips,Roletype,...,Efficiency_GR_reduc,Efficiency_GR_worth,Efficiency_GR_maint,Efficiency_GR_resid,Efficiency_GR_people,Tenure_SC,Education_SC,Discipline_MS,Discipline_MS_other_TEXT,Party_SC
0,100203,20,107,.,56450,Pleasanton City,KS,2056450.0,20107.0,County/Municipal Policymaker,...,4.0,5.0,4.0,1.0,2.0,2.0,6.0,5,NaN,5.0
1,100799,20,173,.,44200,City Of Maize,KS,2044200.0,20173.0,County/Municipal Policymaker,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,108545,20,173,.,73250,City Of Valley Center,KS,2073250.0,20173.0,County/Municipal Policymaker,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,111722,20,043,.,76000,Wathena City,KS,2076000.0,20043.0,County/Municipal Policymaker,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,113753,20,091,.,19825,City Of Edgerton,KS,2019825.0,20091.0,County/Municipal Policymaker,...,2.0,2.0,2.0,2.0,1.0,2.0,5.0,3,NaN,6.0


In [3]:
# doing the variables names in laowercase and all the spaces and all changing with_
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(".", "_", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
    )
    return df

arise = clean_columns(arise)
civic = clean_columns(civic)

print("ARISE columns:")
print(arise.columns.tolist())

print("\nCivicPulse columns:")
print(civic.columns.tolist())

ARISE columns:
['unnamed:_0', 'responseid', 'recipientlastname', 'externalreference', '1st_dis', '2nd_dis', '3rd_dis', 'q3_10_agri', 'q3_9_cyber', 'q3_19_dam', 'q3_1_droughts', 'q3_12_earthquake', 'q3_2_heat', 'q3_3_floods', 'q3_4_ice', 'q3_7_waste_spill', 'q3_8_industrial_fire', 'q3_22_soil_erosion', 'q3_21_severe_storms', 'q3_17_terrorism', 'q3_5_tornados', 'q3_11_infrastructure_failure', 'q3_6_wildfires', 'q3_23_others', 'q3_23_text', 'q4_1', 'q4_2', 'q4_3', 'q4_4', 'q5', 'q6_7_text', 'q7', 'q8', 'q12_1', 'q12_2', 'q12_3', 'q12_4', 'q12_5', 'q12_6', 'q12_7', 'q119', 'q29', 'q30', 'q31', 'q6_1', 'q6_2', 'q6_3', 'q6_4', 'q6_5', 'q6_6', 'q6_7', 'q9_early_warning', 'q9_evacuation_plan', 'q9_financial_assistance_for_low_income_ac', 'q9_water_conservation_programs', 'q9_energy_conservation_programs', 'q9_zoning', 'q9_financial_assistance_for_low_income_shut_offs', 'q9_heating_or_cooling_stations', 'q9_tornado_shelter', 'q9_early_warning_lang', 'q9_code_enforcement', 'q9_backup_electric', 

In [4]:
# Basic Celaning of both datasets 
def basic_cleaning(df):
    df = df.copy()

    # Droping full empty rows and columns
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    # Replace common missing values
    missing_values = ["NA", "N/A", "na", "n/a", "", " ", "None", "null", "did not respond"]
    df = df.replace(missing_values, np.nan)

    # Removing duplicate rows
    df = df.drop_duplicates()

    # Strip spaces from text columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace("nan", np.nan)

    return df

arise = basic_cleaning(arise)
civic = basic_cleaning(civic)

print("ARISE cleaned shape:", arise.shape)
print("CivicPulse cleaned shape:", civic.shape)

ARISE cleaned shape: (309, 89)
CivicPulse cleaned shape: (135, 68)


In [5]:
# keeping the same name in both datasets for response id
arise = arise.rename(columns={"responseid": "id"})
civic = civic.rename(columns={"respondentid": "id"})

In [6]:
# counting the missing values in both datasets
print("ARISE missing values per column:")
display(arise.isnull().sum().sort_values(ascending=False).head(89))

print("Total missing values in ARISE dataset:")
print(arise.isnull().sum().sum())


print("\nCivicPulse missing values per column:")
display(civic.isnull().sum().sort_values(ascending=False).head(68))

print("Total missing values in CivicPulse dataset:")
print(civic.isnull().sum().sum())

ARISE missing values per column:


q3_19_dam                      306
q3_8_industrial_fire           305
q3_7_waste_spill               305
q3_10_agri                     302
q3_23_others                   301
                              ... 
q9_early_warning_lang            0
q9_code_enforcement              0
q9_backup_electric               0
q9_evacuation_route_or_plan      0
q13_none_of_the_above            0
Length: 89, dtype: int64

Total missing values in ARISE dataset:
5533

CivicPulse missing values per column:


mode_ms_other_text           134
leaders_ms_other_text        133
experiments_ms_other_text    132
impacts_ms_other_text        129
disaster_ms_other_text       129
                            ... 
govtype                        0
statecode                      0
stateabb                       0
countycode                     0
id                             0
Length: 68, dtype: int64

Total missing values in CivicPulse dataset:
3003


In [7]:
#Common Variable columns in both datasets
common_vars_map = {

"workforce_challenge_level": ["q8","workforce_sc"],
"fiscal_condition_overall": ["q119","condition_sc"],

"service_early_warning_available": ["q9_early_warning","services_gr_early"],
"service_backup_electric_available": ["q9_backup_electric","services_gr_backup"],
"service_evacuation_plan_available": ["q9_evacuation_route_or_plan","services_gr_evac"],
"service_utility_shutoff_assistance_available": ["q9_financial_assistance_for_low_income_shut_offs","services_gr_financial"],
"service_heating_cooling_stations_available": ["q9_heating_or_cooling_stations","services_gr_heating"],

"early_warning_multilingual": ["q9_early_warning_lang","early_sc"],
"evacuation_support_no_car_households": ["q9_evacuation_plan","evacuation_sc"],

"respondent_tenure_local_gov": ["q29","tenure_sc"],
"respondent_education_level": ["q30","education_sc"],

"equity_group_elderly_support": ["q10_elderly_people","groups_gr_elderly"],
"equity_group_low_income_support": ["q10_low_income","groups_gr_low"],
"equity_group_homeless_support": ["q10_homeless","groups_gr_homeless"],
"equity_group_non_english_support": ["q10_non_english","groups_gr_non"],

# Infrastructure finance variables
"infra_finance_spending_gap": ["q13_reduce_services","infra_gr_spending"],
"infra_finance_unstable_funding": ["q13_defer_capital_projects","infra_gr_relies"],
"infra_finance_budget_constraints": ["q13_reduce_fund_balance","infra_gr_budget"]

}

In [8]:
#Checking results above variables are present in both dataset or not
check_results = []

for harm_var, cols in common_vars_map.items():

    arise_col = cols[0]
    civic_col = cols[1]

    check_results.append({

        "harmonized_variable": harm_var,
        "arise_column": arise_col,
        "arise_exists": arise_col in arise.columns,
        "civic_column": civic_col,
        "civic_exists": civic_col in civic.columns

    })

check_df = pd.DataFrame(check_results)

display(check_df)

,harmonized_variable,arise_column,arise_exists,civic_column,civic_exists
0,workforce_challenge_level,q8,True,workforce_sc,True
1,fiscal_condition_overall,q119,True,condition_sc,True
2,service_early_warning_available,q9_early_warning,True,services_gr_early,True
3,service_backup_electric_available,q9_backup_electric,True,services_gr_backup,True
4,service_evacuation_plan_available,q9_evacuation_route_or_plan,True,services_gr_evac,True
5,service_utility_shutoff_assistance_available,q9_financial_assistance_for_low_income_shut_offs,True,services_gr_financial,True
6,service_heating_cooling_stations_available,q9_heating_or_cooling_stations,True,services_gr_heating,True
7,early_warning_multilingual,q9_early_warning_lang,True,early_sc,True
8,evacuation_support_no_car_households,q9_evacuation_plan,True,evacuation_sc,True
9,respondent_tenure_local_gov,q29,True,tenure_sc,True


In [9]:
# Saving the crosswalk table.
check_df.to_excel("harmonization_crosswalk_table.xlsx", index=False)

print("Crosswalk table saved successfully.")

Crosswalk table saved successfully.


In [10]:
# Created Harmonized varibles in both datasets
for harm_var, cols in common_vars_map.items():
    arise_col = cols[0]
    civic_col = cols[1]

    arise[harm_var] = arise[arise_col]
    civic[harm_var] = civic[civic_col]

In [11]:
#Checked the consistency in both datasets 
harm_vars = list(common_vars_map.keys())

print("Total Harmonized Variables:", len(harm_vars))

print("\nARISE harmonized columns:")
display(arise[harm_vars].head())

print("\nCIVIC harmonized columns:")
display(civic[harm_vars].head())

Total Harmonized Variables: 18

ARISE harmonized columns:


,workforce_challenge_level,fiscal_condition_overall,service_early_warning_available,service_backup_electric_available,service_evacuation_plan_available,service_utility_shutoff_assistance_available,service_heating_cooling_stations_available,early_warning_multilingual,evacuation_support_no_car_households,respondent_tenure_local_gov,respondent_education_level,equity_group_elderly_support,equity_group_low_income_support,equity_group_homeless_support,equity_group_non_english_support,infra_finance_spending_gap,infra_finance_unstable_funding,infra_finance_budget_constraints
0,3.0,3.0,0,1,0,0,0,0,0,3.0,6.0,0,0,0,0,0,1,0
1,4.0,4.0,1,1,1,0,0,0,0,3.0,5.0,0,0,0,0,1,1,0
2,5.0,4.0,1,1,0,1,0,0,0,6.0,6.0,0,0,0,0,1,1,0
3,3.0,3.0,1,1,0,1,1,0,0,3.0,6.0,1,1,0,0,0,1,0
4,3.0,3.0,1,0,0,0,0,0,0,4.0,6.0,0,0,0,0,1,1,1



CIVIC harmonized columns:


,workforce_challenge_level,fiscal_condition_overall,service_early_warning_available,service_backup_electric_available,service_evacuation_plan_available,service_utility_shutoff_assistance_available,service_heating_cooling_stations_available,early_warning_multilingual,evacuation_support_no_car_households,respondent_tenure_local_gov,respondent_education_level,equity_group_elderly_support,equity_group_low_income_support,equity_group_homeless_support,equity_group_non_english_support,infra_finance_spending_gap,infra_finance_unstable_funding,infra_finance_budget_constraints
0,4.0,2.0,1,1,5,5,1,2.0,NaN,2.0,6.0,1.0,2.0,1.0,3.0,2.0,2.0,1.0
1,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,3.0,5.0
2,2.0,4.0,1,1,1,1,1,1.0,3.0,NaN,NaN,3.0,3.0,3.0,4.0,2.0,3.0,2.0
3,4.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4.0,1.0
4,1.0,5.0,3,1,3,"1,3,4",1,3.0,3.0,2.0,5.0,4.0,3.0,1.0,2.0,5.0,4.0,4.0


In [12]:
# Adding common Variable columns in actual civicpulse dataset
for harm_var, cols in common_vars_map.items():

    civic_col = cols[1]

    civic[harm_var] = civic[civic_col]

print("Updated CIVIC dataset shape:", civic.shape)
display(civic.head())

Updated CIVIC dataset shape: (135, 86)


,id,statecode,countycode,countysubcode,placecode,govname,state,stpl_fips,stco_fips,roletype,...,evacuation_support_no_car_households,respondent_tenure_local_gov,respondent_education_level,equity_group_elderly_support,equity_group_low_income_support,equity_group_homeless_support,equity_group_non_english_support,infra_finance_spending_gap,infra_finance_unstable_funding,infra_finance_budget_constraints
0,100203,20,107,.,56450,Pleasanton City,KS,2056450.0,20107.0,County/Municipal Policymaker,...,NaN,2.0,6.0,1.0,2.0,1.0,3.0,2.0,2.0,1.0
1,100799,20,173,.,44200,City Of Maize,KS,2044200.0,20173.0,County/Municipal Policymaker,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,3.0,5.0
2,108545,20,173,.,73250,City Of Valley Center,KS,2073250.0,20173.0,County/Municipal Policymaker,...,3.0,NaN,NaN,3.0,3.0,3.0,4.0,2.0,3.0,2.0
3,111722,20,043,.,76000,Wathena City,KS,2076000.0,20043.0,County/Municipal Policymaker,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4.0,1.0
4,113753,20,091,.,19825,City Of Edgerton,KS,2019825.0,20091.0,County/Municipal Policymaker,...,3.0,2.0,5.0,4.0,3.0,1.0,2.0,5.0,4.0,4.0


In [13]:
#Adding common variable columns in actual arise dataset
for harm_var, cols in common_vars_map.items():

    arise_col = cols[0]

    arise[harm_var] = arise[arise_col]

print("Updated ARISE dataset shape:", arise.shape)
display(arise.head())

Updated ARISE dataset shape: (309, 107)


,unnamed:_0,id,recipientlastname,externalreference,1st_dis,2nd_dis,3rd_dis,q3_10_agri,q3_9_cyber,q3_19_dam,...,evacuation_support_no_car_households,respondent_tenure_local_gov,respondent_education_level,equity_group_elderly_support,equity_group_low_income_support,equity_group_homeless_support,equity_group_non_english_support,infra_finance_spending_gap,infra_finance_unstable_funding,infra_finance_budget_constraints
0,1,R_2sQNn7iItA9jdBn,chief administrator,2079950,3.0,21.0,1.0,NaN,NaN,NaN,...,0,3.0,6.0,0,0,0,0,0,1,0
1,2,R_2Qyp1YFLCR17M9C,chief administrator,2061250,11.0,3.0,8.0,NaN,NaN,NaN,...,0,3.0,5.0,0,0,0,0,1,1,0
2,3,R_2aQAihYklmvMFey,chief administrator,2005600,11.0,21.0,1.0,NaN,NaN,NaN,...,0,6.0,6.0,0,0,0,0,1,1,0
3,4,R_6sTVkGKJ3zy6aPL,chief administrator,2034300,21.0,1.0,2.0,NaN,NaN,NaN,...,0,3.0,6.0,1,1,0,0,0,1,0
4,5,R_3sddVtnZ4CSdNk0,chief administrator,2053225,3.0,1.0,23.0,NaN,NaN,NaN,...,0,4.0,6.0,0,0,0,0,1,1,1


In [14]:
# Saving the new harmonized datesests.
arise.to_excel("arise_harmonized.xlsx", index=False)
civic.to_excel("civicpulse_harmonized.xlsx", index=False)

print("Excel files saved successfully.")

Excel files saved successfully.
